**Tim Pengembang:**

Ahmad Nidzomunnashil - NIM 607012400122

Vikry Achmad Sonjaya - NIM 607012400001

Mardini Dwi Putri - NIM 607012430015

Kelas: 48-02

**Project: Mental Health Risk Prediction**

Notebook ini melatih dan mengevaluasi tiga model klasifikasi (KNN, SVM, Decision Tree) untuk memprediksi tingkat risiko kesehatan mental (0=Rendah, 1=Sedang, 2=Tinggi).

# Load library yang dibutuhkan

In [ ]:
from mpl_toolkits.axisartist.axislines import SubplotZero  # import SubplotZero untuk kebutuhan visualisasi sumbu khusus
import matplotlib.pyplot as plt  # import matplotlib untuk membuat grafik
import numpy as np  # import numpy untuk operasi numerik
import pandas as pd  # import pandas untuk membaca dan mengelola data tabel
import seaborn  # import seaborn untuk mempercantik tampilan grafik
seaborn.set(style='ticks')  # mengatur gaya tampilan grafik seaborn
import matplotlib.cm as cm  # import colormap matplotlib
from sklearn import preprocessing  # import modul preprocessing dari sklearn
from sklearn.preprocessing import MinMaxScaler  # import MinMaxScaler sebagai alternatif normalisasi data
from sklearn.preprocessing import StandardScaler  # import StandardScaler untuk standarisasi fitur
from sklearn.preprocessing import LabelEncoder  # import LabelEncoder untuk mengubah label kategorik menjadi angka
from sklearn.model_selection import train_test_split  # import fungsi untuk membagi data train dan test
from sklearn.model_selection import GridSearchCV  # import GridSearchCV untuk hyperparameter tuning
from sklearn.neighbors import KNeighborsClassifier  # import KNN Classifier
from sklearn.svm import SVC  # import Support Vector Classifier
from sklearn.tree import DecisionTreeClassifier  # import Decision Tree Classifier
from sklearn.metrics import classification_report  # import classification report untuk evaluasi
from sklearn.metrics import accuracy_score  # import fungsi accuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # import fungsi confusion matrix dan visualisasinya

# Load dataset

Dataset diakses dari Google Drive (file id: `1BRSI-0XdgOpB1GUaM3lilUuFrKtRmO0f`). Sel di bawah memakai `gdown` agar dataset langsung ter-download di Colab tanpa perlu mount Drive.

In [ ]:
# Download dataset dari Google Drive
import gdown  # import gdown untuk download dari Google Drive
file_id = '1JvF0t0eg2K7OtcewYpwAXILtjpd3UK78'  # ID file dataset di Google Drive
output_path = 'mental_helth_risk_dataset_1000.csv'  # nama file lokal hasil download
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, quiet=False)  # eksekusi download

In [ ]:
# Load dataset ke dalam DataFrame
df = pd.read_csv(output_path)  # membaca file CSV menjadi DataFrame
print('Shape dataset:', df.shape)  # menampilkan jumlah baris dan kolom
df.head()  # menampilkan 5 baris pertama untuk melihat isi data

In [ ]:
# Cek info kolom dan tipe data
df.info()  # menampilkan tipe data tiap kolom dan jumlah non-null

In [ ]:
# Cek statistik deskriptif fitur numerik
df.describe()  # menampilkan summary statistik fitur numerik

In [ ]:
# Cek missing values
print('Total missing values:', df.isnull().sum().sum())  # cek total missing value
df.isnull().sum()[df.isnull().sum() > 0]  # tampilkan kolom yang punya missing value

In [ ]:
# Cek distribusi target (mental_health_risk)
print('Distribusi target:')  # judul output
print(df['mental_health_risk'].value_counts().sort_index())  # menampilkan jumlah tiap kelas
print()
print('Persentase:')  # judul output persentase
print((df['mental_health_risk'].value_counts(normalize=True) * 100).sort_index().round(2))  # menampilkan persentase tiap kelas

In [ ]:
# Visualisasi distribusi target
plt.figure(figsize=(8, 4))  # membuat figure baru
class_counts = df['mental_health_risk'].value_counts().sort_index()  # ambil count tiap kelas
warna = ['#10b981', '#f59e0b', '#ef4444']  # warna untuk Rendah, Sedang, Tinggi
plt.bar(['Rendah (0)', 'Sedang (1)', 'Tinggi (2)'], class_counts.values, color=warna, edgecolor='black')  # plot bar chart
plt.title('Distribusi Mental Health Risk')  # judul grafik
plt.ylabel('Jumlah')  # label sumbu Y
for i, v in enumerate(class_counts.values):  # iterasi tiap bar untuk anotasi angka
    plt.text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')  # tampilkan angka di atas bar
plt.tight_layout()  # rapikan layout
plt.show()  # tampilkan grafik

# Ambil feature dan label dari dataset

Fitur kategorikal di-encode sesuai sifat datanya:
- `education_level` → **ordinal mapping** (High School < Bachelor < Master < PhD) karena ada urutan jenjang
- `gender`, `marital_status`, `employment_status` → **one-hot encoding** karena tidak ada urutan alami

In [ ]:
df_encoded = df.copy()

# Ordinal encoding untuk education_level (terendah → tertinggi)
education_order = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
df_encoded['education_level'] = df_encoded['education_level'].map(education_order)
print(f'education_level (ordinal): {education_order}')

# Frequency encoding untuk kolom kategorik tanpa urutan alami
kolom_freq_encode = ['gender', 'marital_status', 'employment_status']
freq_encoded_cols = []
for col in kolom_freq_encode:
    # Hitung frekuensi tiap kategori
    freq = df_encoded[col].value_counts(normalize=True)
    # Buat nama kolom baru untuk hasil frequency encoding
    new_col_name = f'{col}_freq_encoded'
    # Map frekuensi ke kolom baru
    df_encoded[new_col_name] = df_encoded[col].map(freq)
    freq_encoded_cols.append(new_col_name)
    print(f'Frequency encoded: {col} -> {new_col_name}')
# Hapus kolom asli yang sudah di-encode
df_encoded = df_encoded.drop(columns=kolom_freq_encode)
print(f'\nKolom hasil frequency encoding: {freq_encoded_cols}')

In [ ]:
# Ambil fitur (X) dan label (y)
X = df_encoded.drop(columns=['mental_health_risk'])  # X = semua kolom kecuali target
y = df_encoded['mental_health_risk']  # y = kolom target

print('Shape X:', X.shape)  # menampilkan ukuran data fitur
print('Shape y:', y.shape)  # menampilkan ukuran data target
print('Jumlah fitur:', X.shape[1])  # jumlah fitur input

# Membagi dataset untuk data training (70%) dan data testing (30%)

In [ ]:
# Split data menjadi 70 persen training dan 30 persen testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)  # split data dengan stratifikasi agar proporsi kelas tetap
print(X_train.shape)  # menampilkan ukuran data training
print(X_test.shape)  # menampilkan ukuran data testing

# Feature scaling


In [ ]:
# Standarisasi fitur menggunakan StandardScaler
scaler = StandardScaler()  # membuat objek StandardScaler
scaler.fit(X_train)  # mempelajari rata-rata dan standar deviasi dari data training
X_train = scaler.transform(X_train)  # mengubah data training ke skala standar
X_test = scaler.transform(X_test)  # mengubah data testing dengan skala yang sama (jangan fit ulang)

# Model 1: K-Nearest Neighbors (KNN)

## Membuat dan training classifier KNN

In [ ]:
n_neighbors_values = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27,29, 31, 33,35,37,39,41,43,45,47,49,51,52,55,57,59,61,63,65,67,69,71,73,75,77,79,81,
                      83,85,87,89,91,92,93,95,97,99]
metric_values = ['manhattan', 'euclidean'] # Daftar metrik yang ingin diuji
results = {}

print("Evaluating KNN with different n_neighbors and metrics values:")
for n in n_neighbors_values:
    for metric in metric_values:
        # Initialize KNeighborsClassifier with current n_neighbors and metric
        classifier_knn = KNeighborsClassifier(metric=metric, n_neighbors=n)

        # Train the model
        classifier_knn.fit(X_train, y_train)

        # Calculate accuracy
        akurasi = classifier_knn.score(X_test, y_test)
        results[(n, metric)] = akurasi
        print(f"n_neighbors={n}, metric='{metric}', Accuracy: {akurasi:.4f}")

# Find and print the best n_neighbors and metric
best_params = max(results, key=results.get)
best_n, best_metric = best_params
print(f"\nBest n_neighbors: {best_n}, Best Metric: '{best_metric}' with Accuracy: {results[best_params]:.4f}")

In [ ]:
# Membuat classifier KNN dengan k=89 
classifier_knn = KNeighborsClassifier(n_neighbors=89, metric='manhattan')  # KNN dengan jumlah tetangga = 99 (hasil HPO)

# Melakukan training
classifier_knn.fit(X_train, y_train)  # melatih KNN dengan data training

In [ ]:
# Menghitung akurasi KNN berdasarkan data uji
akurasi_knn = classifier_knn.score(X_test, y_test)  # menghitung akurasi pada data testing
print(f'Tingkat Akurasi KNN: {akurasi_knn * 100:.2f}%')  # menampilkan akurasi dalam persen

In [ ]:
# Confusion matrix KNN
predictions_knn = classifier_knn.predict(X_test)  # prediksi pada data testing
cm_knn = confusion_matrix(y_test, predictions_knn)  # hitung confusion matrix
disp_knn = ConfusionMatrixDisplay(cm_knn, display_labels=['Rendah', 'Sedang', 'Tinggi'])  # buat display
disp_knn.plot(cmap='Blues')  # gambar dengan warna biru
plt.title(f'Confusion Matrix KNN (k=89) - Akurasi: {akurasi_knn*100:.2f}%')  # judul grafik
plt.show()  # tampilkan

# Classification report KNN
print(classification_report(y_test, predictions_knn, target_names=['Rendah', 'Sedang', 'Tinggi']))  # report detail per kelas

## HPO untuk KNN

Mencari nilai k terbaik dengan GridSearchCV.

In [ ]:
# Definisikan grid parameter untuk KNN
param_grid_knn = {  # daftar parameter yang dicoba
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67, 69, 71, 73, 75, 77, 79, 81, 83, 85, 87, 89, 91, 93, 95, 97, 99
],  # variasi nilai k
    'metric': ['euclidean', 'manhattan'],  # variasi metric jarak
}

# GridSearchCV untuk KNN
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, scoring='recall_macro', cv=5, refit=True, verbose=3)  # search parameter terbaik dengan 5-fold CV
grid_knn.fit(X_train, y_train)  # latih dengan semua kombinasi parameter

print('Parameter terbaik KNN:', grid_knn.best_params_)  # tampilkan parameter terbaik
print('Best CV score      :', round(grid_knn.best_score_, 4))  # tampilkan score CV terbaik

In [ ]:
# Menghitung akurasi KNN berdasarkan data uji
akurasi_grid_knn = accuracy_score(y_test, grid_knn.predict(X_test))  # menghitung akurasi pada data testing
print(f'Tingkat Akurasi KNN: {akurasi_grid_knn * 100:.2f}%')  # menampilkan akurasi dalam persen

In [ ]:
# Confusion matrix KNN
predictions_knn = grid_knn.predict(X_test)  # prediksi pada data testing
cm_knn = confusion_matrix(y_test, predictions_knn)  # hitung confusion matrix
disp_knn = ConfusionMatrixDisplay(cm_knn, display_labels=['Rendah', 'Sedang', 'Tinggi'])  # buat display
disp_knn.plot(cmap='Blues')  # gambar dengan warna biru
plt.title(f'Confusion Matrix KNN (k=99) - Akurasi: {akurasi_grid_knn*100:.2f}%')  # judul grafik
plt.show()  # tampilkan

# Classification report KNN
print(classification_report(y_test, predictions_knn, target_names=['Rendah', 'Sedang', 'Tinggi']))  # report detail per kelas

# Model 2: Support Vector Machine (SVM)

## HPO untuk SVM

Memakai GridSearchCV untuk mencari kombinasi C, gamma, dan kernel terbaik.

In [ ]:
# Definisikan parameter range untuk SVC
param_grid_svm = [  # daftar parameter yang akan diuji
    {'C': [1, 10, 100], 'gamma': [0.01, 0.001, 0.0001], 'kernel': ['rbf', 'linear']},  # kombinasi parameter SVC
]

# GridSearchCV untuk SVM
classifier_svm = GridSearchCV(SVC(probability=True), param_grid_svm, scoring='recall_macro', cv=3, refit=True, verbose=3)  # pencarian parameter terbaik
classifier_svm.fit(X_train, y_train)  # melatih model SVC untuk semua kombinasi parameter

print('Parameter terbaik SVM:', classifier_svm.best_params_)  # menampilkan parameter terbaik
print('Best estimator       :', classifier_svm.best_estimator_)  # model terbaik setelah tuning

In [ ]:
# Prediksi data testing dengan SVM
predictions_svm = classifier_svm.predict(X_test)  # memprediksi kelas pada data testing
akurasi_svm = accuracy_score(y_test, predictions_svm)  # menghitung akurasi
print(f'Tingkat Akurasi SVM: {akurasi_svm * 100:.2f}%')  # tampilkan akurasi
print()
print(classification_report(y_test, predictions_svm, target_names=['Rendah', 'Sedang', 'Tinggi']))  # classification report

In [ ]:
# Confusion matrix SVM
cm_svm = confusion_matrix(y_test, predictions_svm, labels=classifier_svm.classes_)  # hitung confusion matrix
disp_svm = ConfusionMatrixDisplay(cm_svm, display_labels=['Rendah', 'Sedang', 'Tinggi'])  # buat display
disp_svm.plot(cmap='Greens')  # gambar dengan warna hijau
plt.title(f'Confusion Matrix SVM - Akurasi: {akurasi_svm*100:.2f}%')  # judul grafik
plt.show()  # tampilkan

##SVM non HPO

### SVM Manual Parameter Evaluation

This section manually evaluates different combinations of C, gamma, and kernel for an SVC model, calculating the macro recall for both the training and testing sets. This allows for a detailed inspection of each combination's performance.

In [ ]:
from itertools import product
from sklearn.metrics import recall_score, accuracy_score

# 1. Definisikan daftar parameter secara terpisah berdasarkan param_grid_svm
list_C = [1, 10, 100]
list_gamma = [0.01, 0.001, 0.0001]
list_kernel = ['rbf', 'linear']

print("Memulai evaluasi manual untuk setiap kombinasi parameter...\n")
print(f"{'C':<5} | {'Gamma':<8} | {'Kernel':<8} | {'Accuracy'}")
print("-" * 40)

# 2. Lakukan looping untuk semua kombinasi menggunakan product()
# total ada 3 x 3 x 2 = 18 kombinasi
for C, gamma, kernel in product(list_C, list_gamma, list_kernel):

    # Inisialisasi model dengan kombinasi parameter saat ini
    # random_state=42 ditambahkan agar hasilnya konsisten (tidak berubah-ubah)
    model = SVC(C=C, gamma=gamma, kernel=kernel, probability=True, random_state=42)

    # Latih model pada data training
    model.fit(X_train, y_train)

    # Lakukan prediksi untuk melihat performanya
    y_pred_test = model.predict(X_test) # Menggunakan X_test & y_test sebagai data validasi

    # Hitung skor
    accuracy = accuracy_score(y_test, y_pred_test)

    # Tampilkan hasil satu per satu ke layar
    print(f"{C:<5} | {gamma:<8} | {kernel:<8} | {accuracy:.4f}")

In [ ]:

# Membuat classifier SVM tanpa HPO (menggunakan nilai parameter manual/tertentu)
classifier_svm_nh = SVC(kernel='rbf', C=100, gamma=0.01, random_state=0)

# Melakukan training
classifier_svm_nh.fit(X_train, y_train) # melatih SVM dengan data training

In [ ]:
predictions_svm_nh = classifier_svm_nh.predict(X_test)  # memprediksi kelas pada data testing
akurasi_svm_nh = accuracy_score(y_test, predictions_svm_nh)  # menghitung akurasi
print(f'Tingkat Akurasi SVM: {akurasi_svm_nh * 100:.2f}%')  # tampilkan akurasi
print()
print(classification_report(y_test, predictions_svm_nh, target_names=['Rendah', 'Sedang', 'Tinggi']))  # classification repo

In [ ]:
# Confusion matrix SVM
cm_svm_nh = confusion_matrix(y_test, predictions_svm_nh, labels=classifier_svm_nh.classes_)  # hitung confusion matrix
disp_svm_nh = ConfusionMatrixDisplay(cm_svm_nh, display_labels=['Rendah', 'Sedang', 'Tinggi'])  # buat display
disp_svm_nh.plot(cmap='Greens')  # gambar dengan warna hijau
plt.title(f'Confusion Matrix SVM - Akurasi: {akurasi_svm_nh*100:.2f}%')  # judul grafik
plt.show()  # tampilkan

### Decision Tree Non-HPO: Looping Seluruh Kombinasi (72 Variasi)
Bagian ini mengevaluasi setiap kemungkinan kombinasi parameter secara eksplisit untuk melihat performa model tanpa otomasi HPO formal.

In [ ]:
from itertools import product

# 1. Definisikan grid parameter untuk Non-HPO
param_grid_non_hpo = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10,  15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

# Membuat daftar semua kombinasi
keys_non = list(param_grid_non_hpo.keys())
combinations_non = list(product(*param_grid_non_hpo.values()))

hasil_loop_non_hpo = []

print(f"Mengevaluasi {len(combinations_non)} kombinasi pada bagian Non-HPO...\n")

# 2. Loop melalui setiap kombinasi
for i, values in enumerate(combinations_non, start=1):
    params = dict(zip(keys_non, values))

    model = DecisionTreeClassifier(
        criterion=params['criterion'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        random_state=0
    )

    # Latih model (menggunakan data non-hpo)
    model.fit(X_train, y_train)

    # Prediksi dan Hitung Akurasi
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Simpan hasil
    entry = params.copy()
    entry['kombinasi'] = i
    entry['accuracy'] = acc
    hasil_loop_non_hpo.append(entry)

    # Tampilkan log setiap kombinasi
    print(f"Kombinasi {i}: {params} -> Akurasi: {acc*100:.2f}%")

#Buat DataFrame hasil
df_hasil_non_hpo = pd.DataFrame(hasil_loop_non_hpo)

#Cari hasil terbaik
terbaik_non_hpo = df_hasil_non_hpo.loc[df_hasil_non_hpo['accuracy'].idxmax()]

print("\n" + "="*50)
print("RINGKASAN NON-HPO")
print("="*50)
print(f"Parameter Terbaik: {terbaik_non_hpo.to_dict()}")
print(f"Akurasi Terbaik: {terbaik_non_hpo['accuracy']*100:.2f}%")

#Tampilkan Tabel
display(df_hasil_non_hpo.sort_values(by='accuracy', ascending=False))

# Model 3: Decision Tree

## HPO untuk Decision Tree

Mencari kombinasi `max_depth`, `min_samples_split`, dan `criterion` terbaik.

In [ ]:
# Definisikan grid parameter untuk Decision Tree
param_grid_dt = {  # daftar parameter yang dicoba
    'criterion': ['gini', 'entropy'],  # kriteria pembagian node
    'max_depth': [5, 10, 15, None],  # kedalaman maksimum tree
    'min_samples_split': [2, 5, 10],  # jumlah minimum sample untuk split node
    'min_samples_leaf': [1, 2, 5],  # jumlah minimum sample dari setiap leaf
}
# GridSearchCV untuk Decision Tree
classifier_dt = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid_dt, scoring='recall_macro', cv=5, refit=True, verbose=1)  # cari parameter terbaik
classifier_dt.fit(X_train, y_train)  # latih dengan semua kombinasi

print('Parameter terbaik DT:', classifier_dt.best_params_)  # parameter terbaik
print('Best estimator      :', classifier_dt.best_estimator_)  # model terbaik

In [ ]:
# Prediksi data testing dengan Decision Tree
predictions_dt = classifier_dt.predict(X_test)  # prediksi kelas
akurasi_dt = accuracy_score(y_test, predictions_dt)  # akurasi
print(f'Tingkat Akurasi DT: {akurasi_dt * 100:.2f}%')  # tampilkan akurasi
print()
print(classification_report(y_test, predictions_dt, target_names=['Rendah', 'Sedang', 'Tinggi']))  # classification report

In [ ]:
# Confusion matrix Decision Tree
cm_dt = confusion_matrix(y_test, predictions_dt, labels=classifier_dt.classes_)  # hitung confusion matrix
disp_dt = ConfusionMatrixDisplay(cm_dt, display_labels=['Rendah', 'Sedang', 'Tinggi'])  # buat display
disp_dt.plot(cmap='Oranges')  # gambar dengan warna oranye
plt.title(f'Confusion Matrix Decision Tree - Akurasi: {akurasi_dt*100:.2f}%')  # judul grafik
plt.show()  # tampilkan

# Visualisasi Decision Tree

In [ ]:
# Visualisasikan struktur Decision Tree
from sklearn.tree import plot_tree  # import fungsi plot tree
plt.figure(figsize=(20, 40))  # ukuran figure besar untuk keterbacaan
plot_tree(  # gambar pohon keputusan
    classifier_dt.best_estimator_,  # model DT terbaik dari GridSearchCV
    max_depth=20,
    feature_names=X.columns.tolist(),  # nama-nama fitur
    class_names=['Rendah', 'Sedang', 'Tinggi'],  # nama kelas
    filled=True,  # warnai node berdasarkan kelas mayoritas
    rounded=True,  # bentuk node bulat
    fontsize=8  # ukuran font
)
plt.title('Decision Tree')  # judul grafik
plt.show()  # tampilkan